# Stage2 / 06 — Branch comparison + export

Walks every stage2 checkpoint dir, reads the per-branch metrics
JSONs, produces a single comparison table (markdown + CSV on Drive),
and optionally exports the best branch to ONNX / TorchScript.

Drives the stage2 → stage3 handoff.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, sys, json, csv
from pathlib import Path
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT); sys.path.insert(0, REPO_ROOT)

In [ ]:
CKPT_ROOT = Path('/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage2/checkpoints')
rows = []
for run_dir in sorted(CKPT_ROOT.glob('*')):
    if not run_dir.is_dir():
        continue
    metrics_json = run_dir / 'best_metrics.json'
    if not metrics_json.exists():
        rows.append({'run': run_dir.name, 'status': 'no metrics yet'})
        continue
    with open(metrics_json) as f:
        m = json.load(f)
    rows.append({'run': run_dir.name, **m})

# Markdown table
keys = sorted({k for r in rows for k in r.keys()})
print('| ' + ' | '.join(keys) + ' |')
print('|' + '|'.join(['---'] * len(keys)) + '|')
for r in rows:
    print('| ' + ' | '.join(str(r.get(k, '')) for k in keys) + ' |')

# CSV dump
out_csv = Path(REPO_ROOT) / 'stage2' / 'metrics' / 'branch_comparison.csv'
out_csv.parent.mkdir(parents=True, exist_ok=True)
with open(out_csv, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=keys)
    w.writeheader()
    for r in rows:
        w.writerow(r)
print('Wrote', out_csv)

### Export hook (TODO — point at the stage2 winner)
Once the table above identifies a winner, reuse the stage1
`notebooks/06_final_train_eval_export.ipynb` export pattern against
the winning checkpoint. Stage2 inherits the `model.predict()` /
demo-contract ONNX path unchanged.